# FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook demonstrates how to explore, process, and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

The dataset contains survey data on mental health indicators, including demographic information and psychometric scores (GAD-7, PHQ-9, PSQ).

### Dataset Source
The dataset is described using the Croissant JSON-LD schema available at:
```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, columns, and their `@id` values.

In [ ]:
# List all RecordSets in the Dataset by @id
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    # Fallback for alternate Croissant Dataset property naming
    record_sets = metadata.recordSet

print('Available Record Sets:')
for rs in record_sets:
    print(f"- {rs['@id']} (name: {rs.get('name', 'N/A')})")
    if 'field' in rs:
        print(f"  Fields for {rs['@id']}:")
        for f in rs['field']:
            print(f"    - {f['@id']} | name: {f.get('name', 'N/A')} [{f.get('dataType', 'unknown')}]" )
    if 'column' in rs:
        print(f"  Columns for {rs['@id']}:")
        for col in rs['column']:
            print(f"    - {col['@id']} | name: {col.get('name', 'N/A')}")
print('\n')
# For demonstration, we will pick the first RecordSet as the main table
if len(record_sets) > 0:
    main_record_set_id = record_sets[0]['@id']
    print(f'Selected main record set for analysis: {main_record_set_id}')
else:
    main_record_set_id = None
    print("No record sets found.")

## 3. Data Extraction

Load the records from each record set into a DataFrame for analysis. All references are by `@id` as defined in the schema.

In [ ]:
# List available RecordSet `@id`s
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        print(f"Loading records from: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"- Columns in {record_set_id}: {df.columns.tolist()}")
    except Exception as e:
        print(f"Failed to load {record_set_id}: {e}")

# Display first few records from the main record set
if main_record_set_id in dataframes:
    print(dataframes[main_record_set_id].head())
else:
    print(f"No data found for main record set {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)

Example: Filter, normalize, and group by fields using actual field `@id`s and column headers.

In [ ]:
# Example EDA steps using a numeric field and a group field (by @id)
# You may adjust these based on the printed RecordSet/fields overview in section 2

# (Suppose the GAD-7 total score field has @id 'gad7_total_score')
numeric_field_id = 'gad7_total_score'  # Replace with actual @id if it differs
# Example: Group by 'gender' (may have @id 'gender')
group_field_id = 'gender'  # Replace as appropriate

if main_record_set_id not in dataframes:
    raise ValueError('No data loaded for main record set.')

df_main = dataframes[main_record_set_id]

if numeric_field_id not in df_main.columns:
    print(f"Field '{numeric_field_id}' not found in main record set. Available columns: {df_main.columns.tolist()}")
else:
    threshold = 7  # Example threshold for GAD-7 indicating moderate anxiety
    filtered_df = df_main[df_main[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize GAD-7 scores
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) 
        / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a demographic field (e.g., gender)
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df)
    else:
        print(f"Group field '{group_field_id}' not available.")

## 5. Visualization

Visualize GAD-7 score distribution and group comparisons.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of GAD-7 scores
if numeric_field_id in df_main.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df_main[numeric_field_id].dropna(), bins=8, kde=True, color='dodgerblue')
    plt.title('Distribution of GAD-7 Total Scores')
    plt.xlabel('GAD-7 Score')
    plt.ylabel('Count')
    plt.show()

# Boxplot by gender if available
if group_field_id in df_main.columns:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df_main, palette='pastel')
    plt.title('GAD-7 Scores by Gender')
    plt.xlabel('Gender')
    plt.ylabel('GAD-7 Score')
    plt.show()

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to:
- Discover and explore the FAIR² Mental Health Survey Dataset via its Croissant schema.
- Review its record sets, fields, and columns (by `@id`).
- Load data into pandas DataFrames for analysis.
- Filter and normalize records for key psychometric scores (e.g., GAD-7), group by demographic fields, and visualize results.

This approach enables robust and reproducible analysis of data structured using AI-ready, FAIR-aligned standards for future machine learning applications.